In [1]:
import altair as alt
import pandas as pd
from pathlib import Path

In [2]:
result_dir = Path("../../clax-results/1-yandex-baseline")

In [3]:
neural_df = pd.concat([pd.read_csv(f) for f in list(result_dir.glob("*/test_*.csv"))], ignore_index=True)
neural_df["method"] = "CLAX (Gradient-based)"
neural_df["model"] = neural_df["model"].str.upper()

em_df = pd.read_csv(result_dir / "test_em.csv")
em_df["method"] = "PyClick (EM-based)"
em_df["model"] = em_df["model"].str.upper()

df = pd.concat([neural_df, em_df])

In [4]:
def sort_order(df, method="", metric=""):
    method_df = df[df["method"] == method]
    method_df = method_df.groupby(["model"]).agg(avg_metric=(metric, "mean")).reset_index()
    return method_df.sort_values(["avg_metric"]).model.to_list()

In [5]:
sort_by_em = sort_order(df, "PyClick (EM-based)", "test_ppl")
sort_by_gradient = sort_order(df, "CLAX (Gradient-based)", "test_ppl")
sort_by_gradient = [m for m in sort_by_gradient if m in sort_by_em]

In [6]:
sort_by_em = sort_order(df, "PyClick (EM-based)", "test_ppl")

ppl_chart = alt.Chart(df, width=250, height=200, title="Perplexity").mark_bar().encode(
    x=alt.X("model", title="", sort=sort_by_em).axis(labelAngle=45),
    xOffset=alt.XOffset("method", title=""),
    y=alt.Y("mean(test_ppl)", title="").scale(zero=False, domain=(1.27, 1.55), clamp=True),
    color=alt.Color("method", title="").scale(scheme="blues"),
) + alt.Chart(df).mark_errorbar(thickness=3).encode(
    x=alt.X("model", sort=sort_by_em).axis(labelAngle=45),
    xOffset=alt.XOffset("method", title=""),
    y=alt.Y("test_ppl", title="").scale(zero=False, domain=(1.27, 1.55), clamp=True),
)

ppl_chart

alt.LayerChart(...)

In [7]:
sort_by_em = sort_order(df, "PyClick (EM-based)", "test_cond_ppl")

cond_ppl_chart = alt.Chart(df, width=250, height=200, title="Conditional PPL").mark_bar().encode(
    x=alt.X("model", title="", sort=sort_by_em).axis(labelAngle=45),
    xOffset=alt.XOffset("method", title=""),
    y=alt.Y("mean(test_cond_ppl)", title="").scale(zero=False, domain=(1.25, 1.55), clamp=True),
    color=alt.Color("method", title="").scale(scheme="blues"),
) + alt.Chart(df).mark_errorbar(thickness=3).encode(
    x=alt.X("model", sort=sort_by_em).axis(labelAngle=45),
    xOffset=alt.XOffset("method", title=""),
    y=alt.Y("test_cond_ppl", title="").scale(zero=False, domain=(1.25, 1.55), clamp=True),
)

cond_ppl_chart

alt.LayerChart(...)

In [8]:
df["train_time_mins"] = df["train_time_s"].map(lambda x: x / 60)

sort_by_em = sort_order(df, "PyClick (EM-based)", "train_time_mins")

training_time_chart = alt.Chart(df, width=250, height=200, title="Training Time (mins)").mark_bar().encode(
    x=alt.X("model", title="", sort=sort_by_em).axis(labelAngle=45),
    xOffset=alt.XOffset("method", title=""),
    y=alt.Y("mean(train_time_mins)", title="").scale(type="log"),
    color=alt.Color("method", title="").scale(scheme="blues"),
        tooltip=["mean(train_time_mins)"]
) + alt.Chart(df).mark_errorbar(thickness=3).encode(
    x=alt.X("model", sort=sort_by_em).axis(labelAngle=45),
    xOffset=alt.XOffset("method", title=""),
    y=alt.Y("train_time_mins", title="").scale(type="log").axis(titlePadding=-5),
)

training_time_chart

alt.LayerChart(...)

In [9]:
chart = (ppl_chart | cond_ppl_chart | training_time_chart).configure_legend(orient="none", legendX=240, legendY=260, direction="horizontal").configure_scale(bandWithNestedOffsetPaddingInner=0.16)
chart

alt.HConcatChart(...)